# AI-асистент продуктового аналітика: демо на реальних даних

Лабораторна робота №4 з МІПА, Бельдяга Назар, ІП-31.

Ідея: **pandas рахує, LLM пояснює.** Усі числа (профіль даних, сегменти, внески, p-value) рахує код,
а Gemini отримує їх у вигляді фактів і пише висновки. Після кожної відповіді код перевіряє, чи немає
в тексті моделі чисел, яких не було у фактах.

Датасет: мобільний застосунок знайомств, 5.04-11.05.2024 (реєстрації, 1,5 млн сесій, платежі).

## 0. Підготовка
Клоную репозиторій і ставлю залежності. Ключ Gemini береться з Colab Secrets (`GEMINI_API_KEY`).

In [1]:
%cd /content
!rm -rf mipa-lab4-product-ai && git clone -q https://github.com/nazarbeldyaga/mipa-lab4-product-ai.git
%cd mipa-lab4-product-ai
!pip install -q -r requirements.txt

/content
/content/mipa-lab4-product-ai


In [2]:
import os
try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except ImportError:
    pass  # локально ключ задається змінною середовища

from product_ai import Analyst, GeminiLLM, load_data, show
import pandas as pd

data = load_data()
for name, df in data.items():
    print(f"{name:10s} {len(df):>9,} рядків, {df.shape[1]} колонок")

users        202,382 рядків, 10 колонок
payments      11,330 рядків, 9 колонок
splits       189,370 рядків, 4 колонок
sessions   1,507,685 рядків, 9 колонок


In [3]:
llm = GeminiLLM()
print("Модель:", llm.model)
ai = Analyst(data, llm)

Модель: gemini-flash-latest


## 1. Автоматичний EDA
Профіль рахує pandas (`profile_table`), модель оцінює якість і пояснює розподіли.

In [4]:
res = ai.eda("sessions")
show(res)

### EDA: sessions (1507685 рядків)
**Оцінка якості: 8/10.** Таблиця sessions містить 1507685 рядків та 9 колонок за період з 2024-04-05 по 2024-05-11. Загалом дані мають високу повноту, рядки поза періодом відсутні (0), але є незначні дублікати та сильний дисбаланс у статі й тривалості сесій.

**Проблеми:**
- 🟢 `table`: Наявність дублікатів повних рядків у таблиці (14)
- 🟠 `duration`: Велика кількість викидів за IQR та наявність сесій із нульовою тривалістю (iqr_outliers: 158104, zeros: 26, max: 71108.0)
- 🟠 `session_number`: Висока частка статистичних викидів та аномально велике максимальне значення (iqr_outliers: 181841, max: 35884.0)
- 🟢 `country_name`: Пропущені значення (3)
- 🟢 `platform`: Пропущені значення та невизначені платформи (missing: 3, Unknown: 163)
- 🟠 `gender`: Пропущені значення та сильний гендерний дисбаланс (missing: 3, m: 1494999)
- 🟢 `age_group`: Пропущені значення (3)

**Розподіли:**
- `duration_min`: Медіана тривалості становить 22.88 хв, тоді як середнє — 30.09 хв, а максимальне сягає 1185.13 хв через довгий правий хвіст.
- `session_number`: Половина сесій має номер до 120.0 (медіана), але середнє становить 694.92, а максимальне значення сягає 35884.0.
- `platform`: Переважна більшість сесій відбувається через Mobile web (1391838 рядків з часткою 92.32%), тоді як iOS має лише 43619, а Android — 19674.
- `gender`: Абсолютна більшість сесій належить користувачам статі 'm' (1494999 із загальною часткою 99.16%), тоді як на категорію 'f' припадає 12683.
- `country_name`: Топ-країною є Ukraine із 345380 сесіями (22.91% від загального обсягу), на другому місці United States of America (139469).
- `age_group`: Найбільш представленою є вікова група 35-44 (473545 сесій, частка 31.41%), найменшою — 18-24 (101143 сесії).
- `session_start_dt`: Найбільш завантаженим днем періоду стало 2024-04-22 із кількістю сесій 100739.

**Що робити далі:**
- Видалити 14 дублікатів рядків із таблиці.
- Дослідити природу сесій із тривалістю 0.0 (26 рядків) та сесій із тривалістю до 71108.0 секунд (1185.13 хвилин).
- Перевірити користувачів із максимальним номером сесії 35884.0 на ознаки ботів або технічних аномалій.
- Обробити 3 рядки з пропусками у колонках country_name, platform, gender та age_group, а також 163 записи зі значенням 'Unknown' у platform.

✅ Перевірка чисел: усі числа у відповіді є у вхідних фактах.

In [5]:
res = ai.eda("payments")
show(res)

### EDA: payments (11330 рядків)
**Оцінка якості: 9/10.** Таблиця містить 11330 транзакцій від 7781 користувача за період з 2024-04-05 по 2024-05-11. Пропущених значень, дублікатів рядків та записів поза межами періоду немає. Дані демонструють високу цілісність, але містять статистичні викиди за сумою чека через фіксовані цінові тарифи.

**Проблеми:**
- 🟠 `amount_usd`: Статистичні викиди за методом IQR через наявність дорожчих покупок (максимум 19.99) (2561 рядок (22.6%))
- 🟢 `gender`: Критичний дисбаланс транзакцій за статтю на користь чоловіків (11119 платежів (98.14%) від статі 'm')

**Розподіли:**
- `amount_usd`: Медіана та 25-й перцентиль рівні 5.99, середній чек становить 8.27, мінімум 5.01, максимум 19.99.
- `feature_type_name`: Основна частка покупок припадає на 'premium' — 9521 транзакція (84.03%), на 'vip' — 1809.
- `platform`: Більшість оплат здійснено через 'Mobile web' — 10166 (89.73%), нативні додатки iOS та Android сумарно мають 569 і 217 оплат.
- `country_name`: Топ-ринком за оплатами є Ukraine з 3245 транзакціями (28.64%), на другому місці United States of America — 1826.
- `age_group`: Найбільш платоспроможні вікові сегменти: '35-44' (3404 оплати, 30.04%) та '25-34' (3400 оплат).
- `pay_dt`: Пік активності припав на 2024-04-28 з кількістю 590 транзакцій.

**Що робити далі:**
- Сегментувати розподіл amount_usd окремо для 'premium' та 'vip', щоб верифікувати причину 2561 викидів.
- Дослідити низьку частку оплат через додатки iOS (569) та Android (217) порівняно з Mobile web (10166).
- Перевірити воронку конверсії для жіночої аудиторії через низьку кількість транзакцій (211).

✅ Перевірка чисел: усі числа у відповіді є у вхідних фактах.

## 2. Генерація гіпотез
pandas рахує 11 «сигналів» з id (F01...), модель вибирає найцікавіші і формулює гіпотези з посиланням на факти.

In [6]:
res_h = ai.hypotheses(n=7)
show(res_h)

### Гіпотези для перевірки

**1. Онбординг і швидкі офери для нових користувачів** (пріоритет: висока; факти: F11)  
68.38% платників платять у першу 1 годину, а медіана становить 0.2 години, що вказує на вирішальну роль першої сесії для конверсії 4.73% нових користувачів.  
*Чому важливо:* Переважна більшість монетизації відбувається миттєво після реєстрації; оптимізація перших хвилин дасть максимальний приріст оплат.  
*Як перевірити:* Провести A/B тест з промо-пропозицією зі знижкою протягом перших 15 хвилин після реєстрації проти стандартного пейволу.

**2. Оптимізація воронки оплати на Mobile web** (пріоритет: висока; факти: F01)  
Mobile web охоплює 183765 активних користувачів, але має конверсію лише 3.8% та виручку 0.45 USD проти 13.16% та 2.07 USD на Android.  
*Чому важливо:* Mobile web — наймасовіша платформа за користувачами (183765 осіб), тому навіть невеликий ріст конверсії суттєво змінить виручку.  
*Як перевірити:* Порівняти UX чекауту, доступні платіжні методи та швидкість завантаження платіжної сторінки між Mobile web та Android.

**3. Аналіз відтоку активності після піку DAU** (пріоритет: висока; факти: F10)  
Після стрибка з 1802 до 23265 користувачів у день старту A/B тесту 2024-04-19 та піку 33071, DAU впало на -63.9% до 11923.  
*Чому важливо:* Різке падіння DAU після масштабного припливу загрожує втратою залученої бази та зниженням доходів.  
*Як перевірити:* Перевірити когортний ретеншн за днями реєстрації з 2024-04-19 та порівняти метрики утримання між тестовими групами A/B тесту.

**4. Локалізація та ціноутворення в регіонах з низькою конверсією** (пріоритет: середня; факти: F04)  
Конверсія в Україні становить 8.72% з доходом 0.92 USD на активного, тоді як в Еквадорі — 1.04% і 0.12 USD, а в Перу — 1.87% і 0.18 USD при базі 11604 користувачів.  
*Чому важливо:* Країни Латинської Америки формують суттєву аудиторію, але показують конверсію значно нижчу від середньої (3.96%).  
*Як перевірити:* Перевірити наявність локальних платіжних систем і провести тест регіональних цін у Перу та Еквадорі.

**5. Спеціальні промо-акції у вихідні дні** (пріоритет: середня; факти: F08)  
Індекс виручки в неділю досягає 1.125, а в суботу 1.118 відносно середнього, тоді як у вівторок падає до 0.857.  
*Чому важливо:* Вихідні демонструють найвищу платіжну активність відносно середнього 7-денного рівня.  
*Як перевірити:* Запустити пуш-повідомлення з обмеженими за часом знижками у суботу та неділю й порівняти виручку з контрольними вихідними.

**6. Стимулювання VIP-підписки серед аудиторії iOS** (пріоритет: середня; факти: F05, F09)  
Частка VIP-покупок на iOS становить 11.25% проти 33.18% на Android і 15.97% загалом по продукту.  
*Чому важливо:* Сегмент iOS налічує 7887 користувачів і 569 платежів, відстаючи за часткою дорожчих підписок від інших платформ.  
*Як перевірити:* Змінити позиціонування та візуальний акцент на VIP-пакеті на екрані вибору тарифу для iOS.

**7. Збільшення частоти сесій для аудиторії 18-24** (пріоритет: низька; факти: F02, F06)  
Користувачі 18-24 років мають найнижчу кількість сесій на користувача (2.85 проти 15.4 у 55+) і найменшу виручку (0.37 USD проти 0.52 USD у 25-34).  
*Чому важливо:* Сегмент 18-24 досить великий (35459 користувачів), але демонструє слабку залученість і монетизацію.  
*Як перевірити:* Протестувати додаткові інтерактивні механіки та тригерні сповіщення для повернення в застосунок користувачів віком 18-24.

✅ Перевірка чисел: усі числа у відповіді є у вхідних фактах.

In [7]:
# факти, на які посилаються гіпотези (щоб можна було перевірити руками)
pd.DataFrame([{"id": f["id"], "topic": f["topic"]} for f in res_h["facts"]])

,id,topic
0,F01,"Частка платників серед активних, розріз: платф..."
1,F02,"Частка платників серед активних, розріз: вік"
2,F03,"Частка платників серед активних, розріз: стать"
3,F04,Топ-10 країн за активними користувачами: частк...
4,F05,"Тривалість і частота сесій, розріз: платформа"
5,F06,"Тривалість і частота сесій, розріз: вік"
6,F07,Нові реєстрації: перший і останній повний тиждень
7,F08,Виручка за днями тижня відносно ковзного 7-ден...
8,F09,Частка VIP-покупок серед платежів за платформо...
9,F10,Денні активні користувачі (DAU): різкі стрибки...


## 3. Питання природною мовою → pandas
Код показується разом з результатом. Перед запуском його перевіряє `check_code` (без імпортів, файлів, eval).

In [8]:
show(ai.ask("Покажи топ-10 країн за виручкою у квітні 2024 і кількість платників у кожній"))

### Питання: Покажи топ-10 країн за виручкою у квітні 2024 і кількість платників у кожній

Код фільтрує платежі за квітень 2024 року, групує їх за країною та обчислює загальну виручку й кількість унікальних платників, повертаючи топ-10 країн за виручкою.

```python
apr_payments = payments[(payments['pay_dt'] >= pd.Timestamp('2024-04-01')) & (payments['pay_dt'] < pd.Timestamp('2024-05-01'))]
result = (apr_payments.groupby('country_name')
          .agg(revenue=('amount_usd', 'sum'), payers_count=('user_id', 'nunique'))
          .sort_values(by='revenue', ascending=False)
          .head(10)
          .reset_index())
```

|    | country_name                                         |   revenue |   payers_count |
|---:|:-----------------------------------------------------|----------:|---------------:|
|  0 | Ukraine                                              |  13490.6  |           1254 |
|  1 | United States of America                             |  11695.4  |            713 |
|  2 | South Africa                                         |   3434.45 |            335 |
|  3 | Chile                                                |   3218.89 |            269 |
|  4 | Spain                                                |   2458.52 |            204 |
|  5 | United Kingdom of Great Britain and Northern Ireland |   2176.35 |            170 |
|  6 | Kazakhstan                                           |   2154.2  |            213 |
|  7 | Germany                                              |   1478    |             87 |
|  8 | Argentina                                            |   1184.08 |            113 |
|  9 | Peru                                                 |   1144.77 |            119 |

In [9]:
show(ai.ask("Яка середня тривалість сесії в хвилинах по вікових групах для iOS і Mobile web?"))

### Питання: Яка середня тривалість сесії в хвилинах по вікових групах для iOS і Mobile web?

Фільтруємо сесії для платформ iOS та Mobile web, відкидаємо невизначені вікові групи і розраховуємо середню тривалість сесії у хвилинах для кожної платформи та вікової групи.

```python
filtered_sessions = sessions[sessions['platform'].isin(['iOS', 'Mobile web']) & sessions['age_group'].notna()]
result = filtered_sessions.groupby(['platform', 'age_group'])['duration_min'].mean().reset_index().rename(columns={'duration_min': 'avg_duration_min'}).sort_values(by=['platform', 'age_group'])
```

|    | platform   | age_group   |   avg_duration_min |
|---:|:-----------|:------------|-------------------:|
|  0 | Mobile web | 18-24       |            29.6305 |
|  1 | Mobile web | 25-34       |            30.6844 |
|  2 | Mobile web | 35-44       |            30.3266 |
|  3 | Mobile web | 45-54       |            29.5789 |
|  4 | Mobile web | 55+         |            28.8809 |
|  5 | Mobile web | nan         |            21.0667 |
|  6 | iOS        | 18-24       |            31.464  |
|  7 | iOS        | 25-34       |            31.68   |
|  8 | iOS        | 35-44       |            33.6401 |
|  9 | iOS        | 45-54       |            32.5382 |
| 10 | iOS        | 55+         |            33.3989 |
| 11 | iOS        | nan         |            24.5806 |

In [10]:
# нелогічне питання: у даних немає такої інформації
show(ai.ask("Яка була погода в Києві в день з найбільшою виручкою?"))

### Питання: Яка була погода в Києві в день з найбільшою виручкою?

⚠️ Модель відмовилась відповідати: У наданих даних немає інформації про погоду або дані щодо міст (зокрема, Києва).

## 4. Пояснення зміни метрики
Активних користувачів за тиждень 4-10 травня стало помітно менше, ніж тижнем раніше. Розкладаю зміну за платформою, країною, статтю і віком.

In [11]:
res_c = ai.explain("active_users", base=("2024-04-27", "2024-05-03"), target=("2024-05-04", "2024-05-10"))
show(res_c)

### Зміна метрики «активні користувачі (мали сесію)»: 90 977 → 66 132 (-27.31%)
2024-04-27 - 2024-05-03 проти 2024-05-04 - 2024-05-10
**Кількість активних користувачів знизилася на 24845.0 (-27.31%) з 90977.0 до 66132.0.**

**Головні драйвери:**
- платформа / Mobile web: Падіння на 23832.0 користувачів (rate: -23832.0), що забезпечило 95.9% всієї негативної динаміки.
- країна / Ukraine: Зниження на 7227.0 користувачів (з 12811.0 до 5584.0), частка у зміні — 29.1%.
- вік / 35-44: Зменшення на 8538.0 користувачів (з 27944.0 до 19406.0), частка у зміні — 34.4%.
- вік / 45-54: Зменшення на 6935.0 користувачів (з 18080.0 до 11145.0), частка у зміні — 27.9%.

**Гіпотези причин:**
- Проблеми з трафіком або технічні збої на платформі Mobile web.. *Доказ:* 95.9% загального падіння припадає на Mobile web (внесок -23832.0 із -24845.0).. *Перевірити:* Перевірити обсяги платного та органічного залучення на Mobile web, а також технічну стабільність (помилки, доступність) веб-версії.
- Зниження активності або скорочення маркетингових кампаній в Україні.. *Доказ:* Ukraine показала найбільше падіння серед окремих країн (внесок -7227.0, share_of_change_pct 29.088%).. *Перевірити:* Зіставити маркетингові бюджети та канали залучення в Україні між періодами 2024-04-27 - 2024-05-03 та 2024-05-04 - 2024-05-10.

**Застереження:**
- Оскільки метрика є сумою, ефект mix для всіх розрізів дорівнює 0.0, а весь внесок сформовано rate-компонентою (абсолютною зміною сегментів).
- У розрізі статі представлені лише чоловіки ('m' — 100.0% аудиторії та зміни); дані щодо інших статей відсутні у фактах.
- Значна частина спаду за країнами (34.55%, внесок -8584.0) припадає на групу 'інші (184)', деталізація по яких відсутня.

✅ Перевірка чисел: усі числа у відповіді є у вхідних фактах.

In [12]:
# той самий розклад, але для метрики-відношення: частка платників серед активних (mix + rate)
show(ai.explain("payer_conversion", base=("2024-04-27", "2024-05-03"), target=("2024-05-04", "2024-05-10")))

### Зміна метрики «частка платників серед активних, %»: 2.727 → 3.644 (33.63%)
2024-04-27 - 2024-05-03 проти 2024-05-04 - 2024-05-10
**Частка платників серед активних зросла на 0.917 в.п. (з 2.727% до 3.644%), що становить відносне зростання на 33.63%.**

**Головні драйвери:**
- платформа / Mobile web: Забезпечив 100.068% загального зростання (contribution 0.918), переважно за рахунок підвищення внутрішньої конверсії (rate 0.905 проти mix 0.013).
- країна / Ukraine: Забезпечила 36.564% зміни (contribution 0.335): внутрішня конверсія зросла з 4.91% до 12.16% (rate 0.612), попри падіння частки користувачів з 14.082% до 8.444% (mix -0.277).
- країна / United States of America: Сформували 26.292% зростання (contribution 0.241) завдяки росту конверсії з 2.655% до 4.226% (rate 0.213) та зростанню частки аудиторії з 12.504% до 13.562% (mix 0.028).
- вік / 35-44: Дав 33.092% зміни (contribution 0.304) через ріст внутрішньої конверсії з 2.552% до 3.705% (rate 0.338).

**Гіпотези причин:**
- Зміни у платіжному флоу, ціноутворенні або промо-акції для користувачів Mobile web в Україні.. *Доказ:* В Україні конверсія сегмента підскочила з 4.91% до 12.16% (rate 0.612), при цьому Mobile web дав rate 0.905 на платформі.. *Перевірити:* Перевірити логи релізів платіжного шлюзу та маркетингові кампанії на Mobile web в Україні за період 2024-05-04 - 2024-05-10.
- Відтік або зниження залучення неплатників в Україні, через що частка платників штучно зросла.. *Доказ:* Частка аудиторії України скоротилася з 14.082% до 8.444% (mix -0.277), тоді як конверсія зросла.. *Перевірити:* Порівняти абсолютну кількість активних користувачів (MAU/DAU) та платників в Україні між періодами.
- Покращення монетизації серед вікової групи 35-44 на Mobile web.. *Доказ:* Конверсія вікової групи 35-44 зросла з 2.552% до 3.705% (rate 0.338), давши найбільший внесок серед вікових категорій.. *Перевірити:* Перевірити пропозиції та тарифи, які показувалися сегменту 35-44 у ці дати.

**Застереження:**
- У розрізі статі представлені тільки чоловіки (сегмент 'm' становить 100.0%), дані про інші статі відсутні.
- У фактах наведені лише відносні метрики та частки; абсолютні обсяги активних користувачів і транзакцій відсутні, тому неможливо точно оцінити масштаби зміни бази користувачів.
- Категорія 'інші (184)' країн має значний сукупний внесок 18.25% (contribution 0.167), але не деталізована.

✅ Перевірка чисел: усі числа у відповіді є у вхідних фактах.

## 5. Executive summary

In [13]:
show(ai.summary("2024-05-04", "2024-05-10"))

### Executive summary: 2024-05-04 - 2024-05-10
**Критичне падіння активності та сесій на тлі відносної стабільності доходів і зростання конверсії в платника**

**Що сталося:**
- Активні користувачі скоротилися на -27.31% (з 90977 до 66132), а кількість сесій впала на -47.05% (з 455313 до 241095).
- Виручка зменшилася на -4.76% (з 28357.43$ до 27008.75$), кількість платників знизилася на -2.86% (з 2481 до 2410), а ARPPU змінився на -1.92% (з 11.43$ до 11.21$).
- Частка платників зросла на 33.33% (з 2.73% до 3.64%), а середня тривалість сесії збільшилася на 4.4% (з 29.77 хв до 31.08 хв).
- Кількість нових користувачів дещо просіла на -2.6% (з 40732 до 39674).

**Чому це важливо:** Масштабне просідання загальної активності та кількості сесій сигналізує про проблеми з утриманням або технічні збої, хоча монетизація ядра аудиторії залишається стійкою.

**Наступні кроки:**
- Провести технічний аудит на наявність багів авторизації або збоїв трекінгу сесій та активних користувачів.
- Перевірити когорти ретеншну старих користувачів, оскільки падіння нових користувачів незначне (-2.6%), а активність впала різко.
- Дослідити сегмент лояльних користувачів, які забезпечили зростання тривалості сесій і частки платників.

✅ Перевірка чисел: усі числа у відповіді є у вхідних фактах.